# Sesión 1 · La revolución de IA en finanzas

**Módulo 1: Aplicaciones de Python en finanzas** · Temas 1.1, 1.2 y 1.3 · sábado 14 de noviembre de 2026

Al terminar esta sesión podrás:

1. Explicar qué significa que una máquina "aprenda" y ubicar los LLMs en el mapa de la inteligencia artificial.
2. Verificar con datos una cifra que te dio un LLM, en lugar de creerla.
3. Describir un caso de uso de IA en finanzas con la cadena datos → modelo → decisión → valor.
4. Reconocer en código los dos errores que más dinero han costado: el sesgo de anticipación y el sobreajuste.
5. Entender cómo se organiza un proyecto reproducible (el repositorio de tu equipo).

> **Antes de empezar.** Ve al menú **Archivo → Guardar una copia en Drive**. Trabaja siempre en tu copia.

**Cómo leer este notebook**

- Las celdas con `# Tu código aquí` son ejercicios: ahí escribes tú.
- Después de cada ejercicio hay una celda **Comprueba tu respuesta**.
- En esta sesión hay poco código que escribir y mucho que leer y correr. A partir de la sesión 2 la proporción se invierte.

In [ ]:
# CELDA ESTÁNDAR · Córrela siempre primero.
# Prepara la carpeta de trabajo: en Colab descarga el repositorio de la clase;
# en tu computadora se coloca en la raíz del repositorio.
import os
import subprocess
import sys

REPO = "m1-python-finanzas-2026-2"
URL = f"https://github.com/nana-bug/{REPO}.git"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", URL, f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"], check=False)
    os.chdir(f"/content/{REPO}")
else:
    while not os.path.isdir("src") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Datos de la clase: los mismos archivos para todo el grupo, con fechas fijas.
walmex = pd.read_csv("data/raw/walmex.csv", parse_dates=["fecha"], index_col="fecha")
ipc = pd.read_csv("data/raw/ipc.csv", parse_dates=["fecha"], index_col="fecha")

print("WALMEX:", walmex.index.min().date(), "a", walmex.index.max().date(), "·", len(walmex), "días")
print("IPC:   ", ipc.index.min().date(), "a", ipc.index.max().date(), "·", len(ipc), "días")
walmex.tail(3)

En todo el módulo usamos el **precio de cierre ajustado** (columna `cierre_ajustado`), que ya descuenta dividendos y splits. En la sesión 4 veremos por qué y cuánto cambia el resultado si no se ajusta.

El **rendimiento logarítmico** diario es $r_t = \ln(P_t / P_{t-1})$. Lo estudiaremos con calma en la sesión 3; hoy solo lo usamos.

In [ ]:
# Rendimientos logarítmicos diarios, en el periodo que ambos tienen en común
precios = pd.DataFrame({"walmex": walmex["cierre_ajustado"], "ipc": ipc["cierre"]}).dropna()
rendimientos = np.log(precios).diff().dropna()
rendimientos.describe().loc[["count", "mean", "std", "min", "max"]]

---
## 1 · El mapa de la IA y qué significa aprender (tema 1.1)

**El mapa.** Inteligencia artificial ⊃ aprendizaje automático ⊃ aprendizaje profundo ⊃ modelos de lenguaje (LLMs).

- **Programación tradicional:** una persona escribe las reglas. Datos + reglas → respuestas.
- **Aprendizaje automático:** la máquina encuentra las reglas. Datos + respuestas → reglas.

**Aprender es minimizar una pérdida.** Elegimos una familia de modelos, definimos una medida de error (la pérdida) y buscamos los parámetros que la hacen más pequeña con los datos que tenemos.

Un ejemplo que todo financiero conoce: la **beta** de una acción. El modelo es una recta,

$$r^{\text{WALMEX}}_t = a + b\, r^{\text{IPC}}_t + \varepsilon_t,$$

y la pérdida es el error cuadrático medio. Abajo la máquina "aprende" $a$ y $b$ paso a paso (descenso de gradiente) y luego comparamos con la fórmula cerrada de mínimos cuadrados.

In [ ]:
# Trabajamos en porcentaje para que los números sean cómodos de leer
x = rendimientos["ipc"].to_numpy() * 100
y = rendimientos["walmex"].to_numpy() * 100

a, b = 0.0, 0.0            # empezamos sin saber nada
tasa_de_aprendizaje = 0.1
historial = []

for paso in range(300):
    error = (a + b * x) - y
    perdida = np.mean(error**2)
    historial.append(perdida)
    # El gradiente dice hacia dónde sube la pérdida; nos movemos al lado contrario
    a = a - tasa_de_aprendizaje * 2 * np.mean(error)
    b = b - tasa_de_aprendizaje * 2 * np.mean(error * x)

b_formula, a_formula = np.polyfit(x, y, 1)
print(f"Beta aprendida paso a paso: {b:.4f}")
print(f"Beta con la fórmula cerrada: {b_formula:.4f}")

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(historial)
plt.xlabel("Paso de aprendizaje")
plt.ylabel("Pérdida (error cuadrático medio)")
plt.title("Aprender es bajar la pérdida")
plt.show()

Eso es todo lo que significa "entrenar un modelo": una red neuronal hace exactamente lo mismo, con millones de parámetros en lugar de dos. El desarrollo de modelos de aprendizaje automático corresponde al **Módulo 3**; aquí solo necesitamos la idea.

**Generalizar.** Que la pérdida sea baja con los datos de entrenamiento no garantiza que lo sea con datos nuevos. Esa diferencia es el tema de la sección 5.

---
## 2 · De dónde vienen los LLMs (tema 1.1)

Un modelo de lenguaje aprende una sola tarea: **predecir la siguiente palabra** (en realidad, el siguiente *token*). La historia corta:

1. **Redes recurrentes y LSTM (1997):** leen la secuencia palabra por palabra y guardan una memoria. Sirven también para series de tiempo.
2. **Transformer (2017):** en lugar de leer en orden, cada palabra "atiende" a todas las demás al mismo tiempo. Es el artículo *Attention is all you need*.
3. **LLMs (2018 en adelante):** Transformers enormes entrenados con casi todo el texto disponible.

**Lo que hay que recordar de un LLM**

- No consulta una base de datos: genera el texto más probable. Por eso puede dar una cifra falsa con total seguridad (una *alucinación*).
- No sabe qué pasó después de su fecha de entrenamiento, salvo que busque en internet.
- No hace aritmética de forma confiable si no ejecuta código.

De ahí la regla del módulo: **toda cifra que venga de un LLM se verifica con datos.**

> **Reto (optativo 1).** Las redes LSTM y los modelos fundacionales para series de tiempo se trabajan en un notebook aparte, `01_reto_lstm_modelos_fundacionales.ipynb`, que se agrega a esta carpeta.

---
## 3 · Ejercicio en parejas: verifica una cifra que te dio un LLM (15 minutos)

Trabajen de dos en dos. Una persona pregunta al LLM y la otra escribe el código.

**Paso 1.** Pregunten a un LLM (el que usen):

> ¿Cuál fue la volatilidad anualizada de la acción de Walmart de México (WALMEX) en 2025?

Escriban abajo la cifra que les dio, en decimales (por ejemplo, 22% se escribe `0.22`).

**Paso 2.** Calcúlenla con los datos: la volatilidad anualizada es la desviación estándar de los rendimientos logarítmicos diarios de 2025, multiplicada por $\sqrt{252}$.

Pistas: `rendimientos.loc["2025", "walmex"]` selecciona los rendimientos de 2025; `.std()` calcula la desviación estándar; `np.sqrt(252)` es la raíz de 252.

In [ ]:
CIFRA_LLM = None   # escribe aquí la cifra que te dio el LLM, por ejemplo 0.22

# Tu código aquí
print(f"Volatilidad anualizada de WALMEX en 2025, con datos: {volatilidad_2025:.2%}")
if CIFRA_LLM is not None:
    print(f"Cifra del LLM: {CIFRA_LLM:.2%} · diferencia: {CIFRA_LLM - volatilidad_2025:+.2%}")

In [ ]:
# Comprueba tu respuesta
assert 0.05 < volatilidad_2025 < 1.0, "Una volatilidad anualizada razonable está entre 5% y 100%."
assert np.isclose(volatilidad_2025, rendimientos.loc["2025", "walmex"].std() * np.sqrt(252))
print("Correcto. Pega tu cifra en el chat: todo el grupo debe obtener el mismo número.")

**Paso 3.** Contesten en una línea cada pregunta:

- ¿El LLM dijo de dónde sacó la cifra? ¿Qué periodo y qué tipo de rendimiento usó?
- Si la cifra no coincide, ¿quién tiene la razón y cómo lo demostrarían?

*Escribe aquí tu respuesta.*

---
## 4 · Economía e inteligencia artificial en finanzas (tema 1.2)

### 4.1 El valor de la información y los mercados eficientes

Una predicción vale dinero solo si los demás no la tienen. Si un patrón fuera fácil de ver, muchos comprarían antes de la subida y el precio se ajustaría hoy mismo: el patrón desaparecería. Esa es la idea de la **hipótesis de mercados eficientes**: los precios ya incorporan la información disponible.

Una prueba rápida con el IPC: si el rendimiento de ayer ayudara a predecir el de hoy, la correlación entre ambos sería claramente distinta de cero.

In [ ]:
r_ipc = np.log(ipc["cierre"]).diff().dropna()
correlacion = r_ipc.autocorr(lag=1)
print(f"Correlación entre el rendimiento de ayer y el de hoy (IPC): {correlacion:.3f}")

plt.figure(figsize=(4.5, 4.5))
plt.scatter(r_ipc.shift(1), r_ipc, s=3, alpha=0.3)
plt.xlabel("Rendimiento de ayer")
plt.ylabel("Rendimiento de hoy")
plt.title("IPC: ayer no dice casi nada de hoy")
plt.show()

Predecir rendimientos es difícil porque la señal es débil, no porque falten algoritmos. Por eso la mayor parte del valor de la IA en finanzas ha estado en otros lugares.

### 4.2 Casos reales en la banca global y mexicana

| Institución | Qué hace | Dato | Fuente |
|---|---|---|---|
| Morgan Stanley | Asistente para asesores financieros que consulta unos 100,000 documentos internos | Más de 98% de los equipos de asesores lo usa | [OpenAI](https://openai.com/index/morgan-stanley/) |
| JPMorgan | LLM Suite, asistente interno de trabajo | Unos 50,000 empleados en julio de 2024 | [Fortune](https://fortune.com/2024/07/26/jpmorgan-chatbot-llm-suite-chatgpt-ai-machine-learning-wall-street-banking-finance) |
| Bloomberg | BloombergGPT, modelo entrenado con texto financiero | 50 mil millones de parámetros | [arXiv](https://arxiv.org/abs/2303.17564) |
| BBVA México | Asistente "Blue", que dirige al cliente a la operación que busca | Unos 30 segundos, contra hasta 4 minutos antes | [DPL News](https://dplnews.com/bbva-mexico-asistente-ia-generativa-tiempos-atencion-30-segundos/) |

Observa el patrón: en los cuatro casos la IA **no predice precios**. Ordena información, ahorra tiempo y atiende clientes.

### 4.3 Ejercicio en equipos: diseñen un caso de uso (20 minutos)

En su equipo mixto, elijan un problema de una institución financiera mexicana (banco, afore, aseguradora, casa de bolsa o fintech) y llenen la ficha. Quien tiene perfil financiero propone el problema y la decisión; quien tiene perfil técnico propone los datos y el modelo. Al regresar, cada equipo la presenta en dos minutos.

| | Su respuesta |
|---|---|
| **Problema** (¿qué duele hoy y a quién?) | |
| **Datos** (¿cuáles, de dónde salen, quién los tiene?) | |
| **Modelo** (¿qué predice o qué genera?) | |
| **Decisión** (¿quién hace qué de forma distinta gracias al modelo?) | |
| **Valor** (¿cómo se mide en pesos, tiempo o riesgo?) | |
| **Riesgos** (¿qué puede salir mal?) | |

*Escribe aquí tu respuesta.*

---
## 5 · Datos y modelos para generar valor (tema 1.3)

**La cadena completa:** datos → modelo → decisión → valor. Un modelo que no cambia ninguna decisión no vale nada, por preciso que sea.

Dos errores rompen la cadena sin que nadie lo note, porque hacen que el modelo se vea mejor de lo que es.

### 5.1 Sesgo de anticipación

Ocurre cuando el modelo usa información que **no existía** en el momento de decidir. Veamos una "estrategia" sobre el IPC:

> Si el día fue positivo, estoy invertido.

La trampa está en **cuándo** gano ese rendimiento. Si decido con el rendimiento de hoy, solo puedo estar invertido a partir de mañana. Antes de correr la celda: ¿cuál de las dos versiones crees que gana?

In [ ]:
r = r_ipc.loc["2016":]                             # rendimientos diarios del IPC
senal = (r > 0).astype(int)                        # 1 si el día fue positivo, 0 si no

con_trampa = senal * r                             # gana el mismo rendimiento con el que decidió
correcta = (senal.shift(1) * r).fillna(0)          # decide hoy, gana mañana

resultado = pd.DataFrame({
    "Con sesgo de anticipación": np.exp(con_trampa.cumsum()),
    "Correcta": np.exp(correcta.cumsum()),
    "Comprar y mantener": np.exp(r.cumsum()),
})
resultado.plot(logy=True, figsize=(8, 4), title="Crecimiento de 1 peso invertido en el IPC")
plt.ylabel("Valor (escala logarítmica)")
plt.show()
resultado.iloc[-1].round(2)

La única diferencia entre las dos estrategias es un `shift(1)`. La versión con trampa nunca pierde porque solo "invierte" los días que ya sabe que subieron. En la sesión 5 aprenderás a usar `shift` para no caer en esto.

**Regla práctica:** si un resultado histórico se ve demasiado bueno, lo primero que hay que buscar es un sesgo de anticipación.

### 5.2 Sobreajuste

Ocurre cuando probamos muchas ideas y nos quedamos con la que mejor salió en los datos históricos. Con suficientes intentos, alguna se verá muy bien por pura suerte.

Abajo inventamos **200 estrategias al azar**: cada día una moneda decide si apostamos a que el IPC sube (+1) o a que baja (−1). Ninguna tiene habilidad. Elegimos la mejor con los datos hasta 2022 y vemos qué hacen todas después.

In [ ]:
generador = np.random.default_rng(2026)            # semilla fija: todos obtenemos lo mismo
monedas = generador.choice([-1, 1], size=(len(r), 200))
estrategias = pd.DataFrame(monedas, index=r.index).mul(r, axis=0)

antes = estrategias.loc[:"2022"].mean() * 252      # rendimiento anual donde elegimos
despues = estrategias.loc["2023":].mean() * 252    # rendimiento anual con datos nuevos

mejor = antes.idxmax()
mejores_20 = antes.nlargest(20).index

print(f"La mejor de 200 estrategias al azar es la número {mejor}.")
print(f"Su rendimiento anual hasta 2022 (donde la elegimos): {antes[mejor]:+.1%}")
print(f"Su rendimiento anual desde 2023 (datos nuevos):      {despues[mejor]:+.1%}")
print(f"Las 20 mejores hasta 2022, en promedio, desde 2023:  {despues[mejores_20].mean():+.1%}")
print(f"Las 200, en promedio, desde 2023:                    {despues.mean():+.1%}")
print(f"Correlación entre el antes y el después:             {antes.corr(despues):+.2f}")

In [ ]:
plt.figure(figsize=(5, 4.5))
plt.scatter(antes, despues, s=12, alpha=0.6)
plt.scatter(antes[mejor], despues[mejor], color="firebrick", s=60, label="La 'mejor' hasta 2022")
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
plt.xlabel("Rendimiento anual hasta 2022")
plt.ylabel("Rendimiento anual desde 2023")
plt.title("200 estrategias al azar: antes contra después")
plt.legend()
plt.show()

Hasta 2022 la "mejor" estrategia parece muy buena, pero es una moneda: se ve bien en ese periodo precisamente porque la elegimos ahí. La nube de puntos lo confirma: haber ganado antes no dice nada sobre ganar después. Si la punta roja quedara arriba con datos nuevos, también sería suerte.

**Regla práctica:** un resultado solo cuenta si se mide con datos que no se usaron para elegir el modelo.

---
## 6 · Anatomía de un proyecto (taller)

Para que un análisis valga, otra persona tiene que poder repetirlo y obtener lo mismo. Por eso cada equipo trabaja en un **repositorio** con la misma estructura (la plantilla *cookiecutter data science*, en versión reducida):

```
repositorio-del-equipo/
├── README.md        qué pregunta contestan, con qué activo y quiénes son
├── data/
│   ├── raw/         datos tal como se descargaron; nunca se editan a mano
│   ├── interim/     datos a medio limpiar
│   └── processed/   datos listos para analizar
├── notebooks/       un notebook por práctica, numerados
├── src/             sus funciones (finanzas.py): lo que se reutiliza va aquí
└── tests/           pruebas de esas funciones
```

**Buenas prácticas que pediremos todo el módulo**

1. Fechas fijas en toda descarga, para que el resultado no dependa del día en que se corre.
2. Semilla fija en todo lo que use números aleatorios.
3. Lo que se usa dos veces se convierte en función, con su explicación (docstring) y su prueba.
4. El notebook debe correr de principio a fin con **Entorno de ejecución → Reiniciar sesión y ejecutar todo**.
5. Si usaron un LLM, lo declaran en el README y pueden explicar cada línea.

**Tarea: Práctica 1.** El enunciado está en [`practicas/practica_01`](https://github.com/nana-bug/m1-python-finanzas-2026-2/blob/main/practicas/practica_01/README.md). Entrega: viernes 20 de noviembre de 2026, 23:59.

---
## Cierre

**Lo que te llevas hoy**

- Aprender es minimizar una pérdida; lo difícil es que el resultado sirva con datos nuevos.
- Un LLM genera texto probable, no consulta hechos: toda cifra se verifica con datos.
- En finanzas la IA ha generado más valor ordenando información que prediciendo precios.
- Sesgo de anticipación y sobreajuste hacen que un modelo se vea mejor de lo que es.

**Lecturas opcionales** (no se evalúan)

- Goodfellow, Bengio y Courville (2016), [*Deep Learning*](https://www.deeplearningbook.org), capítulo 1.
- Vaswani et al. (2017), [*Attention is all you need*](https://arxiv.org/abs/1706.03762).